# MNIST 实验：环境与数据准备

在 `experiment/MNIST` 打开本 notebook。

- **优先**：若 `data/raw/` 下已有四个官方 gzip（与 `inspect_mnist.ipynb` 相同），**不会**再联网下载；后面的分类 notebook 也优先走这里。
- **否则**：才会用 `torchvision`，数据落在 `./mnist/`（其内部约定是 `mnist/MNIST/raw/`，与扁平的 `data/raw` 不是同一路径）。

**本文件**：检查四文件是否存在、看一张图的张量形状。分类从 `01_logistic_regression.ipynb` 开始。


## 依赖

```bash
pip install torch torchvision numpy scikit-learn
```


In [1]:
from mnist_from_raw import DATA_RAW, raw_files_available

print("本地原始目录:", DATA_RAW.resolve())
print("四文件齐全:", raw_files_available())


本地原始目录: /data1/zdguo/document-parsing/alextools/experiment/MNIST/data/raw
四文件齐全: True


## 训练集一张样本（`float` 像素约在 `[0,1]`，形状 `1×28×28`）


In [2]:
import torch

from mnist_from_raw import MNISTNumpyDataset, load_all_numpy, raw_files_available

if raw_files_available():
    tx, ty, _, _ = load_all_numpy()
    ds = MNISTNumpyDataset(tx, ty)
    img0, y0 = ds[0]
    print("(data/raw) image shape:", tuple(img0.shape), "dtype:", img0.dtype)
    print("label:", int(y0))
    print("len(train):", len(ds))
else:
    from pathlib import Path

    import torchvision

    MNIST_ROOT = Path("./mnist")
    download = not MNIST_ROOT.is_dir() or not any(MNIST_ROOT.iterdir())
    train_data = torchvision.datasets.MNIST(
        root=str(MNIST_ROOT),
        train=True,
        transform=torchvision.transforms.ToTensor(),
        download=download,
    )
    img0, y0 = train_data[0]
    print("(torchvision) image shape:", tuple(img0.shape), "dtype:", img0.dtype)
    print("label:", int(y0))
    print("len(train):", len(train_data))


(data/raw) image shape: (1, 28, 28) dtype: torch.float32
label: 5
len(train): 60000


/data1/zdguo/document-parsing/alextools/experiment/MNIST/mnist_from_raw.py:78: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  x = torch.from_numpy(np.ascontiguousarray(img)).unsqueeze(0).float().div_(255.0)


## 测试集样本数


In [3]:
from mnist_from_raw import load_all_numpy, raw_files_available

if raw_files_available():
    _, _, _, ty = load_all_numpy()
    print("len(test):", len(ty))
else:
    from pathlib import Path

    import torchvision

    MNIST_ROOT = Path("./mnist")
    download = not MNIST_ROOT.is_dir() or not any(MNIST_ROOT.iterdir())
    test_data = torchvision.datasets.MNIST(
        root=str(MNIST_ROOT),
        train=False,
        transform=torchvision.transforms.ToTensor(),
        download=download,
    )
    print("len(test):", len(test_data))


len(test): 10000


## 小结

- **本地**：`data/raw/*.gz` + 模块 `mnist_from_raw.py`。
- **下载**：`./mnist/` 由 torchvision 管理。

各 `0x_*.ipynb` 会自动 `raw_files_available()` 优先读本地。
